# LOOP ENGINEERING

| Loop Pattern          | What causes another iteration?         | Main purpose                |
| --------------------- | -------------------------------------- | --------------------------- |
| **Retry Loop**        | Failure                                | Try the same task again     |
| **Reflection Loop**   | Need for improvement                   | Self-critique and improve   |
| **Validation Loop**   | Validation failure                     | Satisfy predefined rules    |
| **Evaluation Loop**   | Low evaluation score                   | Improve quality             |
| **Tool Loop**         | More information/tool needed           | Gather external information |
| **Human-in-the-Loop** | Human rejection                        | Human approval/control      |
| **Agent Loop**        | Agent decides another action is needed | Autonomous task completion  |


<div align="center">
    <img src="images/loop.png" width="300" height="700">
</div>

<h1 style="color: #7C3AED;">1. RETRY LOOP</h1>

**A Retry Loop repeats the same LLM task when the generated answer does not satisfy a basic requirement.**

<div align="center">
    <img src="images/retry.png" width="300" height="700">
</div>

In [14]:
from pypdf import PdfReader
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
pdf_path = "HopeAI.pdf"
reader = PdfReader(pdf_path)
pdf_text = ""
for page in reader.pages:
    pdf_text += page.extract_text() + "\n"

print("PDF loaded successfully")

client = OpenAI()
MODEL = "gpt-4o-mini"

# USER QUESTION
question = input("Enter your question: ")
#print("\nQuestion:", question)

PDF loaded successfully


Enter your question:  What courses does Hope AI offer?


In [15]:
for attempt in range(1, 4):
    prompt = f"""
Use the PDF information below to answer the question.

PDF:
{pdf_text}

Question:
{question}

Give a clear answer in at least 2 sentences.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    answer = response.choices[0].message.content
    print(f"\nAttempt {attempt}")
    print(answer)

    # Simple validation
    if len(answer.strip()) > 50:
        print("\nRetry successful.")
        break
    print("\nAnswer too short. Retrying...")


Attempt 1
Hope AI offers comprehensive courses in Artificial Intelligence and Data Science, including topics such as Python Basics, Machine Learning, Deep Learning, Natural Language Processing, and Time Series Analysis. The curriculum also includes advanced AI topics like Transformer Architecture and AI Agents, alongside practical training and mentorship to help students build real-world applications.

Retry successful.


<h1 style="color: #7C3AED;">2. REFLECTION LOOP</h1>

**A Reflection Loop asks the LLM to generate an answer and then critically review its own answer before improving it.**

<div align="center">
    <img src="images/reflection.png" width="300" height="700">
</div>

In [16]:
answer = ""
for iteration in range(1, 4):
    if iteration == 1:
        prompt = f"""
Use the PDF below to answer the question.

PDF:
{pdf_text}

Question:
{question}

Give a clear and accurate answer.
"""
    else:
        prompt = f"""
Improve the previous answer using the PDF.

PDF:
{pdf_text}

Question:
{question}

Previous Answer:
{answer}

Identify weaknesses in the previous answer and provide
a clearer and more accurate answer.
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    answer = response.choices[0].message.content
    print(f"\nIteration {iteration}")
    print(answer)


Iteration 1
Hope AI offers a range of courses focused on Artificial Intelligence and Data Science. The courses cover the following topics:

1. **Python Basics**
   - Control Statements, Functions, Lists, Tuples
   - Libraries: Numpy, Pandas, Matplotlib, Sklearn, Seaborn

2. **Data Science & Analysis**
   - Univariate Analysis, Bivariate Analysis

3. **Machine Learning**
   - Regression (Simple & Multiple Linear Regression, Decision Trees, Random Forest)
   - Classification (Logistic Regression, SVM, Naive Bayes, KNN)
   - Clustering (K-Means, Hierarchical Clustering)

4. **Advanced Machine Learning Techniques**
   - Feature Selection, Dimensionality Reduction (PCA, LDA)

5. **Deep Learning**
   - Neural Networks (ANN, CNN, RNN, LSTMs)
   - Computer Vision (Object Detection, Image Generation, Face Recognition)

6. **Time Series Analysis**
   - Introduction, Visualization, Stationarity, Forecasting Models (ARIMA)

7. **Natural Language Processing**
   - Text Processing, Sentiment Analys

<h1 style="color: #7C3AED;">3. VALIDATION LOOP</h1>

**A Validation Loop generates an answer and checks whether it satisfies predefined rules.**

<div align="center">
    <img src="images/validation.png" width="300" height="700">
</div>

In [17]:
for iteration in range(1, 4):
    prompt = f"""
Use the PDF below.

PDF:
{pdf_text}

Question:
{question}

Answer the question in 2-4 sentences.
Do not invent information.
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    answer = response.choices[0].message.content

    # Validation rules
    valid = True
    if len(answer.strip()) < 50:
        valid = False
    if "I don't know" in answer:
        valid = False
    print(f"\nIteration {iteration}")
    print("Answer:", answer)
    if valid:
        print("Validation: PASS")
        break
    print("Validation: FAIL")


Iteration 1
Answer: Hope AI offers comprehensive courses in Artificial Intelligence and Data Science, covering a range of topics such as Python basics, machine learning (including regression, classification, and clustering), deep learning, time series analysis, and natural language processing. The courses also include advanced topics like cloud computing (AWS, GCP, Azure) and advanced AI concepts (transformer architecture, LLM, AI agents). Additionally, students can work on capstone portfolio projects tailored to their interests, ensuring practical application of their learning.
Validation: PASS


<h1 style="color: #7C3AED;">4. EVALUATION LOOP</h1>

**An Evaluation Loop uses a separate evaluator to judge the generated answer.**

<div align="center">
    <img src="images/evaluation.png" width="300" height="700">
</div>

In [18]:
answer = ""
for iteration in range(1, 4):
    # Generate answer
    prompt = f"""
Use the PDF to answer the question.

PDF:
{pdf_text}

Question:
{question}

Previous Answer:
{answer}

Create an accurate and useful answer.
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    answer = response.choices[0].message.content
    
    # Evaluate answer
    evaluation_prompt = f"""
Evaluate the answer below.

Question:
{question}

Answer:
{answer}

Give a score from 1 to 10.

Return only:

Score: <number>
Feedback: <short feedback>
"""
    evaluation = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": evaluation_prompt}
        ]
    )
    evaluation_text = evaluation.choices[0].message.content
    print(f"\nIteration {iteration}")
    print("Answer:", answer)
    print("\nEvaluation:")
    print(evaluation_text)

    # Simple score extraction
    if "Score: 10" in evaluation_text or "Score: 9" in evaluation_text:
        print("\nEvaluation accepted.")
        break


Iteration 1
Answer: Hope AI offers comprehensive courses in Artificial Intelligence and Data Science, which include the following major topics:

1. **Python Basics**:
   - Control Statements, Functions, Lists, Tuples
   - Libraries: Numpy, Pandas, Matplotlib, Sklearn, Seaborn
   - Data Science Fundamentals: Loading Datasets, Data Cleaning, Feature Scaling

2. **Data Science & Analysis**:
   - Univariate Analysis: Central Tendency, Variance, PDF, CDF, Box Plot, etc.
   - Bivariate Analysis: Covariance, Correlation, Hypothesis Testing

3. **Machine Learning**:
   - Regression: Simple & Multiple Linear Regression, Decision Trees, Random Forest
   - Classification: Logistic Regression, SVM, Naive Bayes, KNN
   - Clustering: K-Means, Hierarchical Clustering

4. **Advanced ML Techniques**:
   - Feature Selection, Dimensionality Reduction: PCA, LDA

5. **Deep Learning**:
   - Neural Networks: ANN, CNN, RNN, LSTMs
   - Computer Vision: Object Detection, Image Generation, Face Recognition

6. 

<h1 style="color: #7C3AED;">5. TOOL LOOP</h1>

**A Tool Loop repeatedly decides whether a tool is required, uses the tool, receives the result, and continues.**

<div align="center">
    <img src="images/tool.png" width="300" height="700">
</div>

In [23]:
def search_pdf(keyword):
    keywords = keyword.lower().split()
    results = []
    for line in pdf_text.split("\n"):
        line_lower = line.lower()

        # Match if any keyword is present
        if any(word in line_lower for word in keywords):
            results.append(line.strip())
    return "\n".join(results[:10])

# TOOL LOOP

context = ""
for iteration in range(1, 4):
    print(f"\nIteration {iteration}")
    prompt = f"""
You are an AI agent answering a question using a PDF.

Original Question:
{question}

Previous Tool Result:
{context}

If you need information from the PDF, return:

SEARCH: <keyword>

If you already have enough information, return:

FINAL: <answer>

Do not return anything else.
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    decision = response.choices[0].message.content.strip()

    print("Agent Decision:")
    print(decision)

    # SEARCH
    if decision.startswith("SEARCH:"):
        keyword = decision.replace("SEARCH:", "").strip()
        print("\nTool Input:")
        print(keyword)
        tool_result = search_pdf(keyword)
        print("\nTool Result:")
        if tool_result:
            print(tool_result)
        else:
            print("No result found.")

        # Give tool result back to agent
        context = tool_result

    elif decision.startswith("FINAL:"):
        final_answer = decision.replace("FINAL:", "").strip()
        print("\nFinal Answer:")
        print(final_answer)
        break


Iteration 1
Agent Decision:
SEARCH: Hope AI courses

Tool Input:
Hope AI courses

Tool Result:
Welcome to Hope AI, an E -Learning platform committed to providing world -class Artificial
Intelligence and Data Science courses. Our goal is to empower individuals passionate about
advancing their knowledge in these fields. Our courses offer comprehensive knowledge and practical
skills in AI and Data Science, supported by mentorship from industry experts.
Founded in 2019, Hope AI started as a startup under the MSME scheme. Since then, we've been
dedicated to offering quality education and training in AI an d Data Science. Proudly, we successfully
Our mission is to empower individuals with the knowledge and skills to succeed in  the AI and Data
Science industry. Through expert mentorship and practical training, we aim to help students achieve
their goals and excel in their careers. We believe everyone can become an AI expert, and we're here
Company Details:

Iteration 2
Agent Decision:
FINAL

<h1 style="color: #7C3AED;">6. HUMAN-IN-THE LOOP</h1>

**A Human-in-the-Loop system pauses the loop and asks a human to approve or modify the result.**

<div align="center">
    <img src="images/human.png" width="300" height="700">
</div>

In [24]:
answer = ""
for iteration in range(1, 4):
    prompt = f"""
Use the PDF below to answer the question.

PDF:
{pdf_text}

Question:
{question}

Previous Answer:
{answer}

Give a clear answer.
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    answer = response.choices[0].message.content
    print(f"\nIteration {iteration}")
    print("\nAI Answer:")
    print(answer)

    # Human decision
    approval = input("\nApprove answer? (yes/no): ").lower()
    if approval == "yes":
        print("\nHuman approved the answer.")
        break
    print("\nHuman rejected the answer.")


Iteration 1

AI Answer:
Hope AI offers courses in Artificial Intelligence and Data Science, covering a variety of topics, which include:

1. **Python Basics**
2. **Data Science Fundamentals**
   - Loading Datasets
   - Data Cleaning
   - Feature Scaling
3. **Data Analysis**
   - Univariate Analysis
   - Bivariate Analysis
4. **Machine Learning**
   - Regression (Simple & Multiple Linear Regression, Decision Trees, Random Forest)
   - Classification (Logistic Regression, SVM, Naive Bayes, KNN)
   - Clustering (K-Means, Hierarchical Clustering)
5. **Advanced Machine Learning Techniques**
   - Feature Selection
   - Dimensionality Reduction (PCA, LDA)
6. **Deep Learning**
   - Neural Networks (ANN, CNN, RNN, LSTMs)
   - Computer Vision (Object Detection, Image Generation, Face Recognition)
7. **Time Series Analysis**
   - Forecasting Models (ARIMA)
8. **Natural Language Processing**
   - Text Processing (Tokenization, Stemming, Lemmatization)
   - Sentiment Analysis
   - Chatbot Creation


Approve answer? (yes/no):  yes



Human approved the answer.


<h1 style="color: #7C3AED;">7. AGENT LOOP</h1>

**An Agent Loop allows an LLM to repeatedly:**<br>
**-Understand the goal**<br>
**-Decide what action to take**<br>
**-Use a tool**<br>
**-Observe the result**<br>
**-Decide the next action**

<div align="center">
    <img src="images/agent.png" width="300" height="700">
</div>

In [25]:
def search_pdf(keyword):
    keywords = keyword.lower().replace('"', '').split()
    lines = pdf_text.split("\n")
    results = []
    for line in lines:
        line_lower = line.lower()
        # Match if at least one keyword is found
        if any(word in line_lower for word in keywords):
            results.append(line.strip())
    return "\n".join(results[:10])

agent_context = ""
for iteration in range(1, 4):
    prompt = f"""
You are a simple AI agent.

Goal:
Answer this question using the PDF.

Question:
{question}

Previous context:
{agent_context}

Available tool:
search_pdf(keyword)

Decide what to do.

Return exactly:

ACTION: SEARCH <keyword>

or

ACTION: FINAL <answer>
"""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    decision = response.choices[0].message.content
    print(f"\nIteration {iteration}")
    print(decision)

    # Tool action
    if "ACTION: SEARCH" in decision:
        keyword = decision.split("ACTION: SEARCH")[1].strip()
        tool_result = search_pdf(keyword)
        print("\nTool Observation:")
        print(tool_result[:500])

        agent_context += f"""
Tool searched: {keyword}
Tool result:
{tool_result}
"""
    # Final action
    elif "ACTION: FINAL" in decision:
        final_answer = decision.split("ACTION: FINAL")[1].strip()
        print("\nFinal Answer:")
        print(final_answer)
        break


Iteration 1
ACTION: SEARCH Hope AI courses

Tool Observation:
Welcome to Hope AI, an E -Learning platform committed to providing world -class Artificial
Intelligence and Data Science courses. Our goal is to empower individuals passionate about
advancing their knowledge in these fields. Our courses offer comprehensive knowledge and practical
skills in AI and Data Science, supported by mentorship from industry experts.
Founded in 2019, Hope AI started as a startup under the MSME scheme. Since then, we've been
dedicated to offering quality education and train

Iteration 2
ACTION: SEARCH Hope AI courses

Tool Observation:
Welcome to Hope AI, an E -Learning platform committed to providing world -class Artificial
Intelligence and Data Science courses. Our goal is to empower individuals passionate about
advancing their knowledge in these fields. Our courses offer comprehensive knowledge and practical
skills in AI and Data Science, supported by mentorship from industry experts.
Founded in 201